# Pipeline v2 — try each stage with real APIs

- **Repo root**: `REPO` should be the `Cats-in-Pain-Bachelors` folder (parent of `data_pipeline_v2/`).
- **`.env`**: load `OPENAI_API_KEY` from `REPO/.env` (used by GPT filter and optional search).
- **GPT filter**: calls `src.gpt_filter.run_gpt_filter` (real OpenAI chat completions).
- **Audio v2**: the full pipeline uses `AudioClassifier` with `best_model.pkl` (binary cat vs non-cat; features from `audio_preclassifier_v2.features`). Section 3 below calls `inference.load_model` / `predict_one` directly for parity checks.

Run notebooks with working directory `data_pipeline_v2/notebooks/` or adjust `REPO` in the first code cell.

In [6]:
from __future__ import annotations

import logging
import sys
import tempfile
from pathlib import Path

from dotenv import load_dotenv

# Repository root (parent of data_pipeline_v2/)
REPO = Path("../..").resolve()
if not (REPO / "data_pipeline_v2" / "src").is_dir():
    REPO = Path.cwd().resolve()
    while REPO != REPO.parent and not (REPO / "data_pipeline_v2" / "src").is_dir():
        REPO = REPO.parent

load_dotenv(REPO / ".env")
# Order: audio_preclassifier_v2 first so `import inference` resolves; `src` still resolves to data_pipeline_v2/src
sys.path.insert(0, str(REPO / "data_pipeline_v2"))
sys.path.insert(0, str(REPO / "audio_preclassifier_v2"))

from src.utils import load_config, project_root

cfg = load_config(REPO / "data_pipeline_v2/config/pipeline.yaml")
# Ensure OpenAI key flows into gpt_filter (load_config already merges env)
cfg.setdefault("gpt_filter", {})["openai_api_key"] = __import__("os").environ.get("OPENAI_API_KEY", "")

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
log = logging.getLogger("try_stages")

print("project_root:", project_root())
print("REPO:", REPO)
print("OPENAI_API_KEY set:", bool(cfg["gpt_filter"].get("openai_api_key")))

project_root: /Users/viktorzozula/Documents/Cats-in-Pain-Bachelors
REPO: /Users/viktorzozula/Documents/Cats-in-Pain-Bachelors
OPENAI_API_KEY set: True


## 1. Tag filter (`filter_by_tags`)
Rule-based metadata filter only.

In [7]:
from src.tag_filter import filter_by_tags

fake = [
    {"video_id": "a1", "title": "Funny cat tv for cats to watch", "category_name": "Pets & Animals", "duration_seconds": 120, "description": "", "tags": [], "channel_title": "Ch"},
    {"video_id": "b2", "title": "Real cat at home playing", "category_name": "Pets & Animals", "duration_seconds": 60, "description": "", "tags": ["cat"], "channel_title": "Ch"},
]
kept, discarded = filter_by_tags(fake, cfg)
print("kept:", len(kept), "discarded:", len(discarded))
for d in discarded:
    print("  discard:", d["video_id"], d.get("discard_reason"))

kept: 1 discarded: 1
  discard: a1 Title keyword match


## 2. GPT filter (`run_gpt_filter`)
Loads `config/gpt_filter_rules.md` as the system prompt and calls **gpt-4o-mini** with your batch. Requires `OPENAI_API_KEY` in `.env`.

In [8]:
from src.gpt_filter import run_gpt_filter

gpt_batch = [
    {
        "video_id": "abc123demo",
        "title": "Kitten at vet clinic examination",
        "description": "Short clip of a real kitten being examined by a veterinarian.",
        "tags": ["cat", "vet", "kitten"],
        "duration_seconds": 45,
        "channel_title": "Animal Clinic",
        "query_language": "English",
        "behavioral_category": "Pain/Vet",
    },
    {
        "video_id": "xyz789demo",
        "title": "Simon's Cat — Full Episode 1",
        "description": "Animated cartoon series.",
        "tags": ["animation", "cartoon"],
        "duration_seconds": 120,
        "channel_title": "Cartoon Channel",
        "query_language": "English",
        "behavioral_category": "Positive_Baseline",
    },
]

with tempfile.TemporaryDirectory() as tmp:
    run_dir = Path(tmp) / "run"
    for sub in ("stage_3_gpt_filter",):
        (run_dir / sub).mkdir(parents=True)
    kept_gpt, discarded_gpt, gstats = run_gpt_filter(gpt_batch, cfg, log, run_dir)

print("GPT stats:", gstats)
print("kept:", [x["video_id"] for x in kept_gpt])
print("discarded:", [x["video_id"] for x in discarded_gpt])

GPT filter: 100%|██████████| 1/1 [00:03<00:00,  3.92s/batch, cost=$0.000, discarded=1, kept=1]
INFO:try_stages:
┌────────────────────────────────────────────────────────┐
│  GPT FILTER SUMMARY (metadata-only filtering)          │
│  Input:          2 candidates                           │
│  Kept:           1 (50.0%)                              │
│  Discarded:      1 (50.0%)                              │
│  High confidence keeps:        1                        │
│  Low confidence keeps:           0 (review recommended)   │
│  Estimated API cost:       $0.000                      │
│  Total tokens used:           857                        │
└────────────────────────────────────────────────────────┘



┌────────────────────────────────────────────────────────┐
│  GPT FILTER SUMMARY (metadata-only filtering)          │
│  Input:          2 candidates                           │
│  Kept:           1 (50.0%)                              │
│  Discarded:      1 (50.0%)                              │
│  High confidence keeps:        1                        │
│  Low confidence keeps:           0 (review recommended)   │
│  Estimated API cost:       $0.000                      │
│  Total tokens used:           857                        │
└────────────────────────────────────────────────────────┘
GPT stats: {'input': 2, 'kept': 1, 'discarded': 1, 'high_conf_keeps': 1, 'low_conf_keeps': 0, 'cost': 0.00012854999999999998, 'tokens': 857}
kept: ['abc123demo']
discarded: ['xyz789demo']


## 3. Audio preclassifier v2 (real `best_model.pkl`)
Uses the same **inference API** as training: `audio_preclassifier_v2/inference.py` (`load_model`, `predict_one`).  
Set `AUDIO_FILE` to any `.wav` / `.mp3` on your machine (e.g. under `data/audio_cat-classification/`).

In [9]:
from inference import load_model, predict_one

MODEL_PATH = REPO / "audio_preclassifier_v2/runs/run_20260408_221318/best_model.pkl"
assert MODEL_PATH.is_file(), f"Missing model: {MODEL_PATH}"

model_v2 = load_model(MODEL_PATH)
print("Loaded:", MODEL_PATH)

# Example path from the training repo layout — change if needed
AUDIO_FILE = REPO / "data/audio_cat-classification/raw/NAYA_DATA_AUG1X/Paining/car_extcoll0163.mp3"
if not AUDIO_FILE.is_file():
    AUDIO_FILE = None
    print("Set AUDIO_FILE to a real audio path on your machine, then re-run this cell.")
else:
    thr = float(cfg.get("audio", {}).get("cat_prob_threshold", 0.5))
    out = predict_one(model_v2, AUDIO_FILE, threshold=thr)
    print(out)

/Users/viktorzozula/Documents/Cats-in-Pain-Bachelors/.venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/viktorzozula/Documents/Cats-in-Pain-Bachelors/.venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator KNeighborsClassifier from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/viktorzozula/Documents/Cats-in-Pain-Bachelors/.venv/lib/python3.13/site-packages/sklearn/base.py:463: Inc

Loaded: /Users/viktorzozula/Documents/Cats-in-Pain-Bachelors/audio_preclassifier_v2/runs/run_20260408_221318/best_model.pkl
{'ok': True, 'path': '/Users/viktorzozula/Documents/Cats-in-Pain-Bachelors/data/audio_cat-classification/raw/NAYA_DATA_AUG1X/Paining/car_extcoll0163.mp3', 'label': 1, 'label_name': 'cat', 'proba_non_cat': 0.41892252281153936, 'proba_cat': 0.5810774771884607, 'threshold': 0.5}


## 4. Pipeline `AudioClassifier` (default v2) vs optional v1 ESC
With default `pipeline.yaml`, `AudioClassifier` loads the v2 pickle and uses **P(cat)** from the binary head (same as section 3). Set `prefer_v2: false` to compare the legacy ESC multi-class voting classifier on the same clip.

In [10]:
from pydub import AudioSegment
from src.audio_classifier import AudioClassifier

ac_v2 = AudioClassifier(cfg, log)
print("Pipeline default:", ac_v2.model_name)

if AUDIO_FILE and AUDIO_FILE.is_file():
    seg = AudioSegment.from_file(str(AUDIO_FILE))[:5000]
    ok2, p2 = ac_v2.predict(seg)
    print("v2 binary: is_cat>=threshold:", ok2, "P(cat):", round(p2, 4))

    cfg_v1 = dict(cfg)
    cfg_v1["audio"] = dict(cfg.get("audio", {}))
    cfg_v1["audio"]["prefer_v2"] = False
    ac_v1 = AudioClassifier(cfg_v1, log)
    ok1, p1 = ac_v1.predict(seg)
    print("v1 ESC (optional): is_cat>=threshold:", ok1, "P(cat class):", round(p1, 4))
else:
    print("Set AUDIO_FILE in the previous cell to run v2 + optional v1 on the same clip.")

/Users/viktorzozula/Documents/Cats-in-Pain-Bachelors/.venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/viktorzozula/Documents/Cats-in-Pain-Bachelors/.venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/viktorzozula/Documents/Cats-in-Pain-Bachelors/.venv/lib/python3.13/site-packages/sklearn/base.py:463: Incon

Pipeline audio wrapper: v1
v1 ESC head: is_cat>=threshold: True P(cat class): 0.7798
